# Laplace Approximation: Logistic Regression

This example opens the *Universal Inference* section. The idea behind the section is simple: if every continuous latent variable is approximated by a Gaussian, any factor that has a log-density can send a message, so inference can run in virtually any model, at the cost of an approximation. The classic way to fit such a Gaussian is the **Laplace approximation**: find the mode of the local log-density and use the curvature at the mode as the precision.

RxInfer does not ship a Laplace approximation yet. Here we build one in about forty lines from three ingredients that RxInfer already has:

1. a **fused stochastic node**: a small distribution type with a `logpdf`, which hides a nonlinear function (here the logistic function) inside a factor;
2. the **rule fallback** `NodeFunctionRuleFallback`, which turns the node's `logpdf` into a message whenever no hand-written rule exists;
3. a **custom form constraint** `LaplaceForm`, which turns any product of such messages into a Gaussian by the Laplace method.

!!! note
    The Laplace machinery below lives in this notebook on purpose. It is a preview of an upcoming generic approximation layer (ExpectationApproximations.jl and a universal rules package), and will move there once it matures.

We test it on the textbook bioassay experiment from *Bayesian Data Analysis* (Gelman et al., 3rd ed., Section 3.7; originally Racine-Poon et al., 1986), where the exact posterior can be computed on a grid, so we can see exactly what the Laplace approximation gets right and what it misses.

In [ ]:
using RxInfer, ReactiveMP, BayesBase, Distributions, LinearAlgebra
using ForwardDiff, SpecialFunctions, StatsFuns, StableRNGs, Plots

## The Laplace toolkit

The Laplace approximation of a density $p(x) \propto \exp(\ell(x))$ is the Gaussian

$$\mathcal{N}\left(x \mid \hat{x}, \left(-\nabla^2 \ell(\hat{x})\right)^{-1}\right),\qquad \hat{x} = \arg\max_x \ell(x).$$

The function `laplace` finds the mode $\hat{x}$ with a damped Newton method and returns the Gaussian. Gradients and Hessians come from automatic differentiation (ForwardDiff.jl), so $\ell$ can be any Julia function.

In [ ]:
function laplace(logp, x0::Real; tol = 1e-10, maxiters = 100)
    x = float(x0)
    d2(z) = ForwardDiff.derivative(w -> ForwardDiff.derivative(logp, w), z)
    for _ in 1:maxiters
        g, h = ForwardDiff.derivative(logp, x), d2(x)
        h < 0 || error("Laplace: the log-density is not concave at x = $x")
        step = -g / h
        t, fx = 1.0, logp(x)
        while logp(x + t * step) < fx - 1e-12 && t > 1e-8 # backtracking line search
            t /= 2
        end
        x += t * step
        abs(t * step) < tol && break
    end
    return NormalMeanVariance(x, -inv(d2(x)))
end

function laplace(logp, x0::AbstractVector; tol = 1e-10, maxiters = 100)
    x = float.(collect(x0))
    for _ in 1:maxiters
        g, H = ForwardDiff.gradient(logp, x), ForwardDiff.hessian(logp, x)
        F = cholesky(Symmetric(-H); check = false)
        issuccess(F) || error("Laplace: the log-density is not concave at x = $x")
        step = F \ g
        t, fx = 1.0, logp(x)
        while logp(x + t * step) < fx - 1e-12 && t > 1e-8 # backtracking line search
            t /= 2
        end
        x += t * step
        norm(t * step) < tol && break
    end
    return MvNormalMeanCovariance(x, Matrix(Symmetric(inv(-ForwardDiff.hessian(logp, x)))))
end

Next, the form constraint. In RxInfer a [form constraint](https://reactivebayes.github.io/ReactiveMP.jl/stable/lib/form-constraints/) is applied to the product of the messages that arrive at a variable. When no closed-form product exists, RxInfer keeps that product as a lazy `ProductOf` object which only knows how to evaluate its `logpdf`; `LaplaceForm` replaces it with its Laplace approximation. Gaussians pass through untouched, so conjugate parts of a model keep their exact updates. The Newton search starts from the mean of the first Gaussian factor in the product, usually the prior, or from zero if the product has no Gaussian factor.

In [ ]:
struct LaplaceForm <: ReactiveMP.AbstractFormConstraint end

ReactiveMP.default_form_check_strategy(::LaplaceForm) = FormConstraintCheckLast()
ReactiveMP.default_prod_constraint(::LaplaceForm) = GenericProd()

ReactiveMP.constrain_form(::LaplaceForm, d::NormalDistributionsFamily) = d
function ReactiveMP.constrain_form(::LaplaceForm, d)
    # A product without a Gaussian factor (a likelihood alone) starts from zero, scalars only
    x0 = something(startpoint(d), 0.0)
    return laplace(x -> logpdf(d, x), x0)
end

startpoint(d::NormalDistributionsFamily) = mean(d)
startpoint(d::ProductOf) = firstpoint((d.left, d.right))
startpoint(d::LinearizedProductOf) = firstpoint(d.vector)
startpoint(d) = nothing

function firstpoint(factors)
    for factor in factors
        x0 = startpoint(factor)
        isnothing(x0) || return x0
    end
    return nothing
end

## The bioassay experiment

Four groups of five animals received a dose $x_i$ (in log g/ml) of a compound, and $y_i$ animals died in group $i$:

In [ ]:
x = [-0.86, -0.30, -0.05, 0.73]
n = [5, 5, 5, 5]
y = [0, 1, 3, 5]

The model is a logistic regression with a Gaussian prior on $\theta = (\alpha, \beta)$:

$$\begin{aligned}
\theta &\sim \mathcal{N}\left(0, \mathrm{diag}(4^2, 20^2)\right),\\
y_i &\sim \mathrm{Binomial}\left(n_i, \sigma(\alpha + \beta x_i)\right),
\end{aligned}$$

where $\sigma$ is the logistic function. The pair $\mathrm{Binomial}$ and $\sigma$ has no conjugate update rules. We fuse them into a single stochastic node, `BinomialLogistic`, whose only job is to evaluate the log-likelihood. The node has the interfaces `out` (the count), `θ`, and the constants `x` and `n`.

In [ ]:
logbinomial(n, k) = loggamma(n + 1) - loggamma(k + 1) - loggamma(n - k + 1)

struct BinomialLogistic{T, X, N} <: DiscreteUnivariateDistribution
    θ::T
    x::X
    n::N
end

function BayesBase.logpdf(d::BinomialLogistic, y)
    η = d.θ[1] + d.θ[2] * d.x
    return -y * log1pexp(-η) - (d.n - y) * log1pexp(η) + logbinomial(d.n, y)
end
BayesBase.insupport(::BinomialLogistic, y) = true

@define_factor_node(node = BinomialLogistic, type = Stochastic, interfaces = [:out, :θ, :x, :n])

In [ ]:
prior_θ = MvNormalMeanCovariance(zeros(2), diagm([4.0^2, 20.0^2]))

@model function bioassay(y, x, n, prior)
    θ ~ prior
    for i in eachindex(y)
        y[i] ~ BinomialLogistic(θ, x[i], n[i])
    end
end

The inference call has two additions to a plain `infer` call: the constraint `q(θ) :: LaplaceForm()`, and the rule fallback. Nothing else is model-specific: no rules, no initialisation, no iterations. The posterior of $\theta$ is the Gaussian prior times four `logpdf` messages, and `LaplaceForm` turns that product into a Gaussian.

In [ ]:
laplace_constraints = @constraints begin
    q(θ)::LaplaceForm()
end

result = infer(
    model = bioassay(x = x, n = n, prior = prior_θ),
    data = (y = y,),
    constraints = laplace_constraints,
    options = (rulefallback = NodeFunctionRuleFallback(),),
)

q_θ = result.posteriors[:θ]

As a sanity check, we apply `laplace` directly to the unnormalised log-posterior, written by hand. Message passing must assemble exactly the same density, so both Gaussians must agree to the tolerance of the Newton search.

In [ ]:
logposterior(θ) = logpdf(prior_θ, θ) + sum(logpdf(BinomialLogistic(θ, x[i], n[i]), y[i]) for i in eachindex(y))

q_direct = laplace(logposterior, zeros(2))

println("max |mean difference|       = ", maximum(abs, mean(q_θ) - mean(q_direct)))
println("max |covariance difference| = ", maximum(abs, cov(q_θ) - cov(q_direct)))

## Exact posterior on a grid

With two parameters, the exact posterior can be evaluated on a fine grid. This is our ground truth.

In [ ]:
αs = range(-4, 6; length = 400)
βs = range(-10, 40; length = 400)
logp_grid = [logposterior([α, β]) for α in αs, β in βs]
p_grid = exp.(logp_grid .- maximum(logp_grid))
p_grid ./= sum(p_grid)

A = [α for α in αs, β in βs]
B = [β for α in αs, β in βs]
grid_mean = [sum(p_grid .* A), sum(p_grid .* B)]
grid_cov = [sum(p_grid .* (A .- grid_mean[1]) .* (A .- grid_mean[1])) sum(p_grid .* (A .- grid_mean[1]) .* (B .- grid_mean[2]));
            sum(p_grid .* (A .- grid_mean[1]) .* (B .- grid_mean[2])) sum(p_grid .* (B .- grid_mean[2]) .* (B .- grid_mean[2]))]

summary_row(name, m, C) = println(rpad(name, 10),
    "E[α] = ", lpad(round(m[1]; digits = 2), 6), "   sd(α) = ", lpad(round(sqrt(C[1, 1]); digits = 2), 5),
    "   E[β] = ", lpad(round(m[2]; digits = 2), 6), "   sd(β) = ", lpad(round(sqrt(C[2, 2]); digits = 2), 5),
    "   corr = ", round(C[1, 2] / sqrt(C[1, 1] * C[2, 2]); digits = 2))

summary_row("grid", grid_mean, grid_cov)
summary_row("Laplace", mean(q_θ), cov(q_θ))

The Laplace approximation is centred at the mode, while the exact posterior of $\beta$ is skewed to the right, so its mean lies above the mode, and its spread is larger than the curvature at the mode suggests. The contour plot shows the same picture: the Laplace ellipse (dashed) matches the core of the posterior, but not the long tail towards large $\beta$. Both densities are drawn relative to their maximum, at the same levels.

In [ ]:
p_laplace = [pdf(q_θ, [α, β]) for α in αs, β in βs]

# Both densities relative to their maximum, drawn at the same levels
levels = [0.05, 0.2, 0.4, 0.6, 0.8]
plt = contour(αs, βs, (p_grid ./ maximum(p_grid))'; levels = levels, seriescolor = :blue, colorbar = false, linewidth = 2,
    xlabel = "α", ylabel = "β", title = "Bioassay posterior", xlims = (-2, 5), ylims = (-5, 30))
contour!(plt, αs, βs, (p_laplace ./ maximum(p_laplace))'; levels = levels, seriescolor = :red, linestyle = :dash, colorbar = false, linewidth = 2)
plot!(plt, [NaN], [NaN]; color = :blue, linewidth = 2, label = "exact (grid)")
plot!(plt, [NaN], [NaN]; color = :red, linestyle = :dash, linewidth = 2, label = "Laplace")
plt

## A derived quantity: the LD50

The quantity of interest in the original study is the LD50, the dose at which the probability of death is 50%, $\mathrm{LD50} = -\alpha / \beta$. It is a nonlinear function of $\theta$, so we compare its distribution under the exact posterior (sampled from the grid) and under the Laplace approximation, keeping only samples with $\beta > 0$ as in the book.

In [ ]:
rng = StableRNG(42)
nsamples = 10_000
idx = rand(rng, Categorical(vec(p_grid)), nsamples)
ld50_grid = [-A[i] / B[i] for i in idx if B[i] > 0]
ld50_laplace = [-θ[1] / θ[2] for θ in eachcol(rand(rng, q_θ, nsamples)) if θ[2] > 0]

histogram(ld50_grid; bins = range(-0.6, 0.6; length = 61), normalize = :pdf, alpha = 0.5, label = "exact (grid)", xlabel = "LD50", title = "Posterior of the LD50")
histogram!(ld50_laplace; bins = range(-0.6, 0.6; length = 61), normalize = :pdf, alpha = 0.5, label = "Laplace")

In [ ]:
println("P(β > 0):  grid = ", round(sum(p_grid[B .> 0]); digits = 4), ",  Laplace = ", round(1 - cdf(Normal(mean(q_θ)[2], sqrt(cov(q_θ)[2, 2])), 0); digits = 4))
println("LD50 median:  grid = ", round(median(ld50_grid); digits = 3), ",  Laplace = ", round(median(ld50_laplace); digits = 3))
println("LD50 90% interval:  grid = ", round.(quantile(ld50_grid, [0.05, 0.95]); digits = 3), ",  Laplace = ", round.(quantile(ld50_laplace, [0.05, 0.95]); digits = 3))

The medians agree, but the Laplace interval for the LD50 is wider than the exact one. The Gaussian is symmetric around the mode of $\beta$, so it puts a few percent of its mass near and below $\beta = 0$, a region the data rule out, and dividing by small values of $\beta$ spreads the LD50 out.

## The same toolkit, another model: Poisson regression

Nothing in `LaplaceForm` knows about logistic regression. To show that, we fit a Poisson regression with a log link, $y_i \sim \mathrm{Poisson}(\exp(a + b t_i))$, on simulated counts. The only new code is the fused node.

In [ ]:
struct PoissonLogLinear{T, X} <: DiscreteUnivariateDistribution
    θ::T
    t::X
end

BayesBase.logpdf(d::PoissonLogLinear, y) = (η = d.θ[1] + d.θ[2] * d.t; y * η - exp(η) - loggamma(y + 1))
BayesBase.insupport(::PoissonLogLinear, y) = true

@define_factor_node(node = PoissonLogLinear, type = Stochastic, interfaces = [:out, :θ, :t])

@model function poisson_regression(y, t, prior)
    θ ~ prior
    for i in eachindex(y)
        y[i] ~ PoissonLogLinear(θ, t[i])
    end
end

θ_true = [0.5, 1.2]
t = collect(range(-1, 1.5; length = 30))
counts = [rand(rng, Poisson(exp(θ_true[1] + θ_true[2] * ti))) for ti in t]

result_poisson = infer(
    model = poisson_regression(t = t, prior = MvNormalMeanCovariance(zeros(2), diagm([10.0, 10.0]))),
    data = (y = counts,),
    constraints = laplace_constraints,
    options = (rulefallback = NodeFunctionRuleFallback(),),
)

q_poisson = result_poisson.posteriors[:θ]
println("true θ = ", θ_true, ",  posterior mean = ", round.(mean(q_poisson); digits = 3), ",  posterior sd = ", round.(sqrt.(diag(cov(q_poisson))); digits = 3))

In [ ]:
ts = range(-1, 1.5; length = 100)
draws = rand(rng, q_poisson, 200)
rates = [exp(θ[1] + θ[2] * τ) for τ in ts, θ in eachcol(draws)]
lower, upper = [quantile(r, 0.05) for r in eachrow(rates)], [quantile(r, 0.95) for r in eachrow(rates)]
middle = [exp(mean(q_poisson)[1] + mean(q_poisson)[2] * τ) for τ in ts]

plot(ts, middle; ribbon = (middle .- lower, upper .- middle), fillalpha = 0.3, label = "posterior rate (90%)", xlabel = "t", ylabel = "count")
plot!(ts, exp.(θ_true[1] .+ θ_true[2] .* ts); color = :black, linestyle = :dash, label = "true rate")
scatter!(t, counts; label = "observed counts", color = :gray)

## Exact vs approximate, and what did not work

- **What worked.** With a fused node, the rule fallback and `LaplaceForm`, RxInfer ran a textbook Laplace approximation in two non-conjugate models without a single hand-written message update rule. The result coincides with a Laplace approximation of the hand-written posterior to numerical precision.
- **Where the approximation shows.** The Laplace approximation is centred at the mode and uses local curvature, so it misses the skew of the bioassay posterior in $\beta$: the posterior mean of $\beta$ is underestimated by about a third. Because a Gaussian is symmetric, it also puts a few percent of its mass on $\beta < 0$, which the data rule out, and this inflates the uncertainty of the LD50. Moment matching (expectation propagation) is the natural next step to capture the mean and spread better; it is on the roadmap for this section.
- **What we had to work around.**
  - The nonlinear function had to be fused into a stochastic node. Writing `y[i] ~ Binomial(n[i], logistic(α + β * x[i]))` creates a deterministic node for `logistic`, and the rule fallback does not apply to deterministic nodes yet.
  - $\theta$ is a single two-dimensional variable. With separate scalar variables $\alpha$ and $\beta$ the fallback would collapse each one to its mean when computing the message for the other, which ignores their strong posterior correlation.
  - The fallback messages have no normalisation, so the Bethe free energy cannot be computed for this model yet.